# Teste — módulo de estimação

**Kernel:** conda `moradinha`  
**Diretório de trabalho:** raiz do projeto (`moradinha/`)  
**Pré-requisito:** DuckDB do município já populado pelo `modulo_coleta` (Grupos 1–6).

Células marcadas com `⏳ Stub` levantam `NotImplementedError` — serão habilitadas conforme
cada etapa for implementada.

In [52]:
import importlib
import logging
import sys
from pathlib import Path


def _encontrar_raiz() -> Path:
    candidato = Path.cwd()
    for _ in range(6):
        if (candidato / "modulo_coleta").is_dir():
            return candidato
        candidato = candidato.parent
    raise FileNotFoundError(
        f"Raiz do projeto não encontrada a partir de '{Path.cwd()}'. "
        "Execute o notebook com o cwd dentro da pasta do projeto."
    )


ROOT = _encontrar_raiz()
print(f"ROOT detectado: {ROOT}")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)

import modulo_estimacao.orquestrador as orch_est
importlib.reload(orch_est)
from modulo_estimacao.orquestrador import estimar_municipio

print("OK")

ROOT detectado: d:\Doutorado\UFABC\moradinha
OK


## Parâmetros

Edite antes de rodar.

In [53]:
CODIGO_IBGE  = "2700300"          # Arapiraca-AL
NOME_PASTA   = "al_arapiraca"     # subpasta dentro de data/processed/
ANO_T0       = 2022
ANO_T1       = 2024
RESOLUCAO_H3 = 8                  # res=8 ≈ 460m de raio por hexágono
ETAPAS       = [1, 2, 3, 4, 5]   # [1..9]; inclua 6,7,8 quando t1 estiver pronto
MODELO_T0    = "rf"               # 'rf' | 'lm' | 'gwr'
MODELO_TEMP  = "lm"               # 'lm' | 'rf'

BASE_DIR = ROOT / "data"
DB_PATH  = BASE_DIR / "processed" / NOME_PASTA / f"{NOME_PASTA}.duckdb"
print(f"DuckDB: {DB_PATH}")
print(f"Existe: {DB_PATH.exists()}")

DuckDB: d:\Doutorado\UFABC\moradinha\data\processed\al_arapiraca\al_arapiraca.duckdb
Existe: True


## Insumos disponíveis no DuckDB

Verifica se todas as tabelas do `modulo_coleta` necessárias estão presentes.

In [54]:
from modulo_coleta.utils.db_utils import abrir_conexao, listar_tabelas

conn = abrir_conexao(DB_PATH)
tabelas = set(listar_tabelas(conn))

INSUMOS_T0 = {
    "setores_censitarios": "Grupo 1 — geometria dos setores",
    "grade_estatistica":   "Grupo 1 — grade IBGE 200m",
    "domicilio01":         "Grupo 2 — domicílios",
    "domicilio02":         "Grupo 2 — moradores por dormitório",
    "responsavel01":       "Grupo 2 — renda dos responsáveis",
    "enderecos_cnefe_residencial":    "Grupo 3 — CNEFE residencial",
    "enderecos_cnefe_naoresidencial": "Grupo 3 — CNEFE não-residencial",
    f"luminosidade_{ANO_T0}": f"Grupo 4 — VIIRS {ANO_T0}",
    "pnadc_deficit_componentes": "Grupo 5 — estimativa PNADc",
    f"mapbiomas_{ANO_T0}": f"Grupo 6 — MapBiomas {ANO_T0}",
    "fcu_setor":           "Grupo 6 — FCU por setor",
}

INSUMOS_T1 = {
    f"luminosidade_{ANO_T1}": f"Grupo 4 — VIIRS {ANO_T1}",
    f"mapbiomas_{ANO_T1}":    f"Grupo 6 — MapBiomas {ANO_T1}",
}

print(f"{'Tabela':<40} {'Status':<8} Origem")
print("-" * 72)
todas_ok = True
for tabela, origem in {**INSUMOS_T0, **INSUMOS_T1}.items():
    ok = tabela in tabelas
    if not ok:
        todas_ok = False
    icone = "✅" if ok else "❌"
    n = conn.execute(f"SELECT COUNT(*) FROM {tabela}").fetchone()[0] if ok else "—"
    print(f"{tabela:<40} {icone:<8} {origem}  [{n}]")

print()
print("Todos os insumos t0 OK" if todas_ok else "⚠ Insumos faltando — execute modulo_coleta primeiro")
conn.close()

21:15:50 [INFO] DuckDB aberto: d:\Doutorado\UFABC\moradinha\data\processed\al_arapiraca\al_arapiraca.duckdb
Tabela                                   Status   Origem
------------------------------------------------------------------------
setores_censitarios                      ✅        Grupo 1 — geometria dos setores  [396]
grade_estatistica                        ✅        Grupo 1 — grade IBGE 200m  [4940]
domicilio01                              ❌        Grupo 2 — domicílios  [—]
domicilio02                              ❌        Grupo 2 — moradores por dormitório  [—]
responsavel01                            ❌        Grupo 2 — renda dos responsáveis  [—]
enderecos_cnefe_residencial              ✅        Grupo 3 — CNEFE residencial  [96193]
enderecos_cnefe_naoresidencial           ✅        Grupo 3 — CNEFE não-residencial  [18845]
luminosidade_2022                        ✅        Grupo 4 — VIIRS 2022  [396]
pnadc_deficit_componentes                ✅        Grupo 5 — estimativa PNADc  [

## Orquestrador (todas as etapas)

Quando todas as etapas estiverem implementadas, esta célula executa o pipeline completo.
Por enquanto levanta `NotImplementedError` — use as células por etapa abaixo para
desenvolvimento incremental.

In [55]:
try:
    resultados = estimar_municipio(
        codigo_ibge=CODIGO_IBGE,
        ano_t0=ANO_T0,
        ano_t1=ANO_T1,
        db_path=DB_PATH,
        resolucao_h3=RESOLUCAO_H3,
        etapas=ETAPAS,
        modelo_t0=MODELO_T0,
        modelo_temporal=MODELO_TEMP,
    )

    # Exibe o status geral
    status = resultados.get("status", "?")
    icone_geral = "✅" if status == "ok" else "❌"
    print(f"{icone_geral} Status geral: {status}")
    print(f"Mensagem: {resultados.get('mensagem', 'N/A')}\n")

    # Exibe detalhes das etapas
    for categoria, etapas in {
        "✅ Executadas com sucesso": resultados.get("etapas_ok", []),
        "❌ Com erro": resultados.get("etapas_erro", []),
        "⏭ Puladas (já existentes)": resultados.get("etapas_puladas", []),
    }.items():
        if etapas:
            print(f"{categoria}: {etapas}")

except NotImplementedError as e:
    print(f"⏳ Orquestrador ainda não implementado.\n{e}")

21:15:50 [INFO] [Orquestrador] municipio=2700300 | t0=2022 | t1=2024 | res_h3=8 | etapas=[1, 2, 3, 4, 5] | forcar=False
21:15:50 [INFO] DuckDB aberto: d:\Doutorado\UFABC\moradinha\data\processed\al_arapiraca\al_arapiraca.duckdb
21:15:50 [INFO] [Orquestrador] Etapas com saídas já salvas (serão puladas): [1, 2, 3, 4, 5]
21:15:50 [INFO] [Orquestrador] Etapa 1 — PULADA (já existe no DuckDB)
21:15:50 [INFO] [Orquestrador] Etapa 2 — PULADA (já existe no DuckDB)
21:15:50 [INFO] [Orquestrador] Etapa 3 — PULADA (já existe no DuckDB)
21:15:50 [INFO] [Orquestrador] Etapa 4 — PULADA (já existe no DuckDB)
21:15:50 [INFO] [Orquestrador] Etapa 5 — PULADA (já existe no DuckDB)
21:15:50 [INFO] [Orquestrador] Concluído — status=ok | Nenhuma etapa executada — 5 já existiam no DuckDB (forcar=False).
✅ Status geral: ok
Mensagem: Nenhuma etapa executada — 5 já existiam no DuckDB (forcar=False).

⏭ Puladas (já existentes): [1, 2, 3, 4, 5]


---
## Sub-pipeline t0 — Etapas 1-5

Células independentes para desenvolvimento e depuração de cada etapa.

### Etapa 1 — Proxy do déficit por setor (Censo 2022)

In [56]:
import importlib
import modulo_estimacao.etapas_t0.etapa1_proxy_setor as e1
importlib.reload(e1)
from modulo_estimacao.etapas_t0.etapa1_proxy_setor import calcular_proxy_setor

conn = abrir_conexao(DB_PATH)
try:
    res1 = calcular_proxy_setor(CODIGO_IBGE, conn)
    print(res1)
except NotImplementedError as e:
    print(f"⏳ Etapa 1 — stub:\n{e}")
finally:
    conn.close()

21:15:50 [INFO] DuckDB aberto: d:\Doutorado\UFABC\moradinha\data\processed\al_arapiraca\al_arapiraca.duckdb
21:15:50 [INFO] [Etapa 1] Iniciando proxy_setor — municipio 2700300
21:15:50 [INFO] [Etapa 1] Carregando tabelas censo
21:15:50 [INFO] [Etapa 1] Calculando proporcoes de carencia
21:15:50 [INFO] [Etapa 1] Carregando renda do responsavel
21:15:50 [INFO] [Etapa 1] Juntando geometria
21:15:50 [WARNING] [Etapa 1] 72 setores sem geometria (16.9% do total)
21:15:50 [INFO] [Etapa 1] 426 setores | 77695 domicilios totais | proxy medio=0.1887
21:15:50 [INFO] [Etapa 1] Salvando proxy_setor no DuckDB
21:15:51 [INFO] [Etapa 1] proxy_setor salvo: 426 linhas
{'status': 'ok', 'n_setores': 426, 'n_sem_geometria': 72, 'n_dom_total': 77695, 'proxy_medio': 0.188736, 'mensagem': 'proxy_setor calculado: 426 setores, 77695 domicilios.'}


In [57]:
# Inspeção do proxy_setor quando Etapa 1 estiver implementada
conn = abrir_conexao(DB_PATH)
try:
    df = conn.execute("""
        SELECT
            COUNT(*)                          AS n_setores,
            SUM(n_dom_total)                  AS dom_total,
            SUM(proxy_carencias_setor)        AS proxy_carencias,
            ROUND(AVG(proxy_carencias_setor), 4) AS taxa_media,
            MAX(proxy_carencias_setor)        AS taxa_max
        FROM proxy_setor
    """).df()
    display(df)
except Exception as e:
    print(f"⏳ proxy_setor ainda não gerado: {e}")
finally:
    conn.close()

21:15:51 [INFO] DuckDB aberto: d:\Doutorado\UFABC\moradinha\data\processed\al_arapiraca\al_arapiraca.duckdb


,n_setores,dom_total,proxy_carencias,taxa_media,taxa_max
0,426,77690.0,79.457812,0.1887,0.666667


### Etapa 2 — Covariáveis territoriais (t0)

In [58]:
import modulo_estimacao.etapas_t0.etapa2_covariaveis_t0 as e2
importlib.reload(e2)
from modulo_estimacao.etapas_t0.etapa2_covariaveis_t0 import (
    extrair_covariaveis_setor_t0,
    extrair_covariaveis_h3_t0,
)

conn = abrir_conexao(DB_PATH)
try:
    res2 = extrair_covariaveis_setor_t0(CODIGO_IBGE, ANO_T0, conn)
    print("covariaveis_setor_t0:", res2)
    res2h = extrair_covariaveis_h3_t0(CODIGO_IBGE, ANO_T0, RESOLUCAO_H3, conn)
    print("covariaveis_h3_t0:", res2h)
except NotImplementedError as e:
    print(f"⏳ Etapa 2 — stub:\n{e}")
finally:
    conn.close()

21:15:51 [INFO] DuckDB aberto: d:\Doutorado\UFABC\moradinha\data\processed\al_arapiraca\al_arapiraca.duckdb
21:15:51 [INFO] [Etapa 2] Extraindo covariaveis_setor_t0 — municipio 2700300, ano 2022
21:15:51 [INFO] [Etapa 2] Carregando base (proxy_setor)
21:15:51 [INFO] [Etapa 2] Luminosidade 2022
21:15:51 [INFO] [Etapa 2] MapBiomas 2022
21:15:51 [INFO] [Etapa 2] FCU
21:15:51 [INFO] [Etapa 2] CNEFE residencial densidade
21:15:52 [INFO] [Etapa 2] CNEFE nao-residencial densidade
21:15:53 [INFO] [Etapa 2] Distancia ao centro
21:15:53 [INFO] [Etapa 2] CNEFE buffer 500 m
21:15:53 [INFO] [Etapa 2] Carregando geometria dos setores
21:15:53 [INFO] [Etapa 2] 426 setores | 222 com todas as covariaveis preenchidas
21:15:54 [INFO] [Etapa 2] covariaveis_setor_t0 salvo: 426 linhas
covariaveis_setor_t0: {'status': 'ok', 'n_setores': 426, 'n_completos': 222, 'mensagem': 'covariaveis_setor_t0: 426 setores, 222 completos.'}
21:15:54 [INFO] [Etapa 2-H3] Agregando para H3 res=8 — municipio 2700300
21:15:54 [I

In [59]:
# Inspeção de correlações entre covariáveis e proxy quando implementado
conn = abrir_conexao(DB_PATH)
try:
    df = conn.execute("""
        SELECT
            CORR(renda_resp_media,          proxy_setor.proxy_carencias_setor) AS r_renda,
            CORR(luminosidade_setor_mean,   proxy_setor.proxy_carencias_setor) AS r_lum,
            CORR(prop_urbano,               proxy_setor.proxy_carencias_setor) AS r_urbano,
            CORR(fcu_area_pct,              proxy_setor.proxy_carencias_setor) AS r_fcu
        FROM covariaveis_setor_t0
        JOIN proxy_setor USING (cod_setor)
    """).df()
    print("Correlações covariável × proxy:")
    display(df)
except Exception as e:
    print(f"⏳ covariaveis_setor_t0 ainda não gerado: {e}")
finally:
    conn.close()

21:15:56 [INFO] DuckDB aberto: d:\Doutorado\UFABC\moradinha\data\processed\al_arapiraca\al_arapiraca.duckdb
Correlações covariável × proxy:


,r_renda,r_lum,r_urbano,r_fcu
0,-0.077667,-0.409758,-0.463819,-0.098493


### Etapa 3 — Modelo espacial

In [60]:
import modulo_estimacao.etapas_t0.etapa3_modelo_espacial as e3
importlib.reload(e3)
from modulo_estimacao.etapas_t0.etapa3_modelo_espacial import ajustar_modelo_t0

conn = abrir_conexao(DB_PATH)
try:
    res3 = ajustar_modelo_t0(CODIGO_IBGE, ANO_T0, conn, modelo=MODELO_T0)
    print(res3)
except NotImplementedError as e:
    print(f"⏳ Etapa 3 — stub:\n{e}")
finally:
    conn.close()

21:15:57 [INFO] DuckDB aberto: d:\Doutorado\UFABC\moradinha\data\processed\al_arapiraca\al_arapiraca.duckdb
21:15:57 [INFO] [Etapa 3] Ajustando modelo 'rf' — municipio 2700300, ano 2022
21:15:57 [INFO] [Etapa 3] Dataset de treino: 222 setores
21:15:58 [INFO] [Etapa 3] RF serializado: data\processed\2700300\modelo_t0_rf_2700300.pkl
21:15:58 [INFO] [Etapa 3] Calculando Moran's I nos residuos
21:15:58 [INFO] [Etapa 3] R²=0.7583 | RMSE=0.0533 | CV(RMSE)=0.265 | Moran's I=0.0345 (p=0.249)
21:15:58 [INFO] [Etapa 3] modelo_t0_diagnostico salvo no DuckDB
{'status': 'ok', 'modelo': 'rf', 'r2': 0.7582712207147818, 'rmse': 0.0532510976396474, 'cv_rmse_y': 0.26502093511385627, 'cv_rmse_5fold': 0.08607129893643421, 'moran_i': 0.034488841506970076, 'moran_p': 0.249, 'n_setores': 222, 'pkl_path': 'data\\processed\\2700300\\modelo_t0_rf_2700300.pkl', 'mensagem': "Modelo 'rf' ajustado: R²=0.7583, RMSE=0.0533."}


In [61]:
# Diagnóstico do modelo: R², RMSE, Moran's I dos resíduos
conn = abrir_conexao(DB_PATH)
try:
    df = conn.execute("SELECT * FROM modelo_t0_diagnostico").df()
    display(df)
    # Diagnóstico espacial: I < 0.10 → sem autocorrelação residual significativa
    moran = df["moran_i_residuos"].iloc[0]
    p     = df["moran_p"].iloc[0]
    msg   = "OK — resíduos sem estrutura espacial" if moran < 0.10 else "⚠ Considerar GWR ou lag espacial"
    print(f"Moran's I dos resíduos = {moran:.3f}  (p={p:.3f})  → {msg}")
except Exception as e:
    print(f"⏳ modelo_t0_diagnostico ainda não gerado: {e}")
finally:
    conn.close()

21:15:59 [INFO] DuckDB aberto: d:\Doutorado\UFABC\moradinha\data\processed\al_arapiraca\al_arapiraca.duckdb


,modelo,r2_treino,rmse_treino,cv_rmse_y,cv_rmse_5fold,moran_i_residuos,moran_p,n_setores,features_usadas,ano_t0,timestamp,feature_importances
0,rf,0.758271,0.053251,0.265021,0.086071,0.034489,0.249,222,"[""renda_resp_media"", ""luminosidade_setor_mean""...",2022,2026-04-28T00:15:58.656780+00:00,"{""luminosidade_setor_mean"": 0.1953460320451385..."


Moran's I dos resíduos = 0.034  (p=0.249)  → OK — resíduos sem estrutura espacial


### Etapa 4 — Predição em H3 (t0)

In [62]:
import modulo_estimacao.etapas_t0.etapa4_predicao_h3_t0 as e4
importlib.reload(e4)
from modulo_estimacao.etapas_t0.etapa4_predicao_h3_t0 import predizer_h3_t0

conn = abrir_conexao(DB_PATH)
try:
    res4 = predizer_h3_t0(CODIGO_IBGE, ANO_T0, RESOLUCAO_H3, conn)
    print(res4)
except NotImplementedError as e:
    print(f"⏳ Etapa 4 — stub:\n{e}")
finally:
    conn.close()

21:15:59 [INFO] DuckDB aberto: d:\Doutorado\UFABC\moradinha\data\processed\al_arapiraca\al_arapiraca.duckdb
21:15:59 [INFO] [Etapa 4] Predição H3 res=8 — municipio 2700300, ano 2022
21:15:59 [INFO] [Etapa 4] Carregando modelo 'rf'
21:15:59 [INFO] [Etapa 4] Carregando covariaveis_h3_t0
21:15:59 [INFO] [Etapa 4] Aplicando modelo a 431 hexágonos
21:15:59 [INFO] [Etapa 4] Gerando geometrias H3
21:15:59 [INFO] [Etapa 4] 431 hexágonos | 428 completos | deficit_estimado_total=14159.8
21:16:00 [INFO] [Etapa 4] deficit_predito_h3_t0 salvo: 431 linhas
{'status': 'ok', 'n_hexagonos': 431, 'n_completos': 428, 'deficit_estimado_total': 14159.8, 'mensagem': 'deficit_predito_h3_t0: 431 hexágonos, deficit_total=14160.'}


### Etapa 5 — Calibração IPF (t0)

In [63]:
import modulo_estimacao.etapas_t0.etapa5_calibracao_t0 as e5
importlib.reload(e5)
from modulo_estimacao.etapas_t0.etapa5_calibracao_t0 import calibrar_h3_t0

conn = abrir_conexao(DB_PATH)
try:
    res5 = calibrar_h3_t0(CODIGO_IBGE, conn)
    print(res5)
except NotImplementedError as e:
    print(f"⏳ Etapa 5 — stub:\n{e}")
finally:
    conn.close()

21:16:00 [INFO] DuckDB aberto: d:\Doutorado\UFABC\moradinha\data\processed\al_arapiraca\al_arapiraca.duckdb
21:16:00 [INFO] [Etapa 5] Calibração t0 — municipio 2700300
21:16:00 [INFO] [Etapa 5] Carregando deficit_predito_h3_t0
21:16:00 [WARNING] [Etapa 5] 28 H3 sem deficit_estimado (n_domicilios_grade=NaN) — excluídos da calibração
21:16:00 [INFO] [Etapa 5] Atribuindo setor dominante a cada H3
21:16:00 [INFO] [Etapa 5] Calculando alvos por setor (proxy_carencias × n_dom)
21:16:00 [INFO] [Etapa 5] Alvo Censo total=13063.1 | Predito RF total=14159.8 | razão=0.923
21:16:00 [INFO] [Etapa 5] Passo 1: distribuição intraurbana por setor (Censo)
21:16:00 [INFO] [Etapa 5] Passo 1 concluído: soma_cal1=5498.0
21:16:00 [WARNING] [Etapa 5] CV PNADc = 73.4% (alto). Âncora aplicada com ressalva. Considere usar_ancora_pnadc=False para preservar apenas a distribuição Censo.
21:16:00 [INFO] [Etapa 5] Passo 2: fator_dominio=0.6819 (PNADc=3749.0, cal1=5498.0)
21:16:00 [INFO] [Etapa 5] Deficit calibrado to

In [64]:
# Consistência pós-IPF: soma dos H3 por setor deve ≈ proxy_setor (delta < 0.01)
conn = abrir_conexao(DB_PATH)
try:
    df = conn.execute("""
        SELECT
            h3.cod_setor,
            SUM(h3.deficit_calibrado) AS soma_h3,
            p.proxy_total_sem_onus,
            ABS(SUM(h3.deficit_calibrado) - p.proxy_total_sem_onus) AS delta_abs
        FROM deficit_calibrado_h3_t0 h3
        JOIN proxy_setor p USING (cod_setor)
        GROUP BY h3.cod_setor, p.proxy_total_sem_onus
        ORDER BY delta_abs DESC
        LIMIT 10
    """).df()
    print("Setores com maior desvio pós-IPF (deve ser < 0.01):")
    display(df)

    # Consistência com PNADc
    pnadc = conn.execute("""
        SELECT deficit_total, cv_deficit_total
        FROM pnadc_deficit_componentes
        WHERE componente = 'deficit_total'
        LIMIT 1
    """).fetchone()
    total_h3 = conn.execute("SELECT SUM(deficit_calibrado) FROM deficit_calibrado_h3_t0").fetchone()[0]
    if pnadc:
        print(f"\nTotal H3 calibrado: {total_h3:.0f}")
        print(f"PNADc déficit total: {pnadc[0]:.0f}  (CV={pnadc[1]:.2f})")
        print(f"Razão H3/PNADc: {total_h3/pnadc[0]:.3f}  (ideal ≈ 1.0)")
except Exception as e:
    print(f"⏳ deficit_calibrado_h3_t0 ainda não gerado: {e}")
finally:
    conn.close()

21:16:02 [INFO] DuckDB aberto: d:\Doutorado\UFABC\moradinha\data\processed\al_arapiraca\al_arapiraca.duckdb
⏳ deficit_calibrado_h3_t0 ainda não gerado: Binder Error: Column "cod_setor" does not exist on left side of join!


---
## Sub-pipeline t1 — Etapas 6-8

Requer sub-pipeline t0 completo e VIIRS/MapBiomas de `ANO_T1` no DuckDB.

### Etapa 6 — Covariáveis t1 + deltas

In [65]:
import modulo_estimacao.etapas_t1.etapa6_covariaveis_t1 as e6
importlib.reload(e6)
from modulo_estimacao.etapas_t1.etapa6_covariaveis_t1 import extrair_covariaveis_setor_t1

conn = abrir_conexao(DB_PATH)
try:
    res6 = extrair_covariaveis_setor_t1(CODIGO_IBGE, ANO_T0, ANO_T1, conn)
    print(res6)
except NotImplementedError as e:
    print(f"⏳ Etapa 6 — stub:\n{e}")
finally:
    conn.close()

21:16:02 [INFO] DuckDB aberto: d:\Doutorado\UFABC\moradinha\data\processed\al_arapiraca\al_arapiraca.duckdb
21:16:02 [INFO] [Etapa 6] Extraindo covariaveis_setor_t1 — municipio 2700300, t0=2022, t1=2024
21:16:02 [INFO] [Etapa 6] Carregando valores t0 de covariaveis_setor_t0
21:16:02 [INFO] [Etapa 6] 426 setores carregados de t0
21:16:02 [INFO] [Etapa 6] Luminosidade 2024 (t1)
21:16:02 [INFO] [Etapa 6] MapBiomas 2024 (t1)
21:16:02 [INFO] [Etapa 6] 426 setores | lum_t1 preenchidos: 223 | MapBiomas_t1 preenchidos: 354
21:16:02 [INFO] [Etapa 6] Setores com flag_expansao (delta_urb > 5pp): 27/426 | delta_lum_dominio (media): 7.7945
21:16:02 [INFO] [Etapa 6] covariaveis_setor_t1 salvo: 426 linhas
21:16:02 [INFO] [Etapa 6] delta_covariaveis_setor salvo: 426 linhas
{'status': 'ok', 'camadas': ['covariaveis_setor_t1', 'delta_covariaveis_setor'], 'n_setores': 426, 'n_expansao': 27, 'delta_lum_dominio_pnadc': 7.794539710361207}


In [66]:
# Distribuição de expansão urbana: quantos setores com flag_expansao=True?
conn = abrir_conexao(DB_PATH)
try:
    df = conn.execute("""
        SELECT
            COUNT(*) FILTER (WHERE flag_expansao) AS n_expansao,
            COUNT(*) AS n_total,
            ROUND(AVG(delta_lum_mean), 4) AS delta_lum_medio,
            ROUND(AVG(delta_prop_urbano), 4) AS delta_urbano_medio
        FROM delta_covariaveis_setor
    """).df()
    display(df)
except Exception as e:
    print(f"⏳ delta_covariaveis_setor ainda não gerado: {e}")
finally:
    conn.close()

21:16:03 [INFO] DuckDB aberto: d:\Doutorado\UFABC\moradinha\data\processed\al_arapiraca\al_arapiraca.duckdb


,n_expansao,n_total,delta_lum_medio,delta_urbano_medio
0,27,426,7.7945,0.0199


### Etapa 7 — Modelo temporal

In [67]:
import modulo_estimacao.etapas_t1.etapa7_modelo_temporal as e7
importlib.reload(e7)
from modulo_estimacao.etapas_t1.etapa7_modelo_temporal import ajustar_modelo_temporal

conn = abrir_conexao(DB_PATH)
try:
    res7 = ajustar_modelo_temporal(CODIGO_IBGE, ANO_T0, ANO_T1, conn, modelo=MODELO_TEMP)
    print(res7)
except NotImplementedError as e:
    print(f"⏳ Etapa 7 — stub:\n{e}")
finally:
    conn.close()

21:16:03 [INFO] DuckDB aberto: d:\Doutorado\UFABC\moradinha\data\processed\al_arapiraca\al_arapiraca.duckdb
21:16:03 [INFO] [Etapa 7] Ajustando modelo temporal 'lm' — municipio 2700300, t0=2022, t1=2024
{'status': 'erro', 'mensagem': "Modelo t0 não encontrado em 'data\\processed\\al_2700300'. Esperado: modelo_t0_rf_2700300.pkl ou modelo_t0_lm_2700300.pkl. Execute a Etapa 3 antes."}


### Etapa 8 — Predição H3 (t1)

In [68]:
import modulo_estimacao.etapas_t1.etapa8_predicao_h3_t1 as e8
importlib.reload(e8)
from modulo_estimacao.etapas_t1.etapa8_predicao_h3_t1 import predizer_h3_t1

conn = abrir_conexao(DB_PATH)
try:
    res8 = predizer_h3_t1(CODIGO_IBGE, ANO_T0, ANO_T1, RESOLUCAO_H3, conn)
    print(res8)
except NotImplementedError as e:
    print(f"⏳ Etapa 8 — stub:\n{e}")
finally:
    conn.close()

21:16:03 [INFO] DuckDB aberto: d:\Doutorado\UFABC\moradinha\data\processed\al_arapiraca\al_arapiraca.duckdb
21:16:03 [INFO] [Etapa 8] Predição H3 t1 res=8 — municipio 2700300, t0=2022, t1=2024
21:16:03 [INFO] [Etapa 8] Carregando baseline t0
21:16:03 [INFO] [Etapa 8] 431 hexágonos H3 carregados de t0
21:16:03 [INFO] [Etapa 8] Carregando delta_proxy por setor (Etapa 7)
21:16:03 [INFO] [Etapa 8] Agregando delta_proxy setor → H3
21:16:03 [INFO] [Etapa 8] 429/431 hexágonos com delta_proxy agregado
21:16:03 [INFO] [Etapa 8] deficit_predito_t1 total=3587.8 (t0=3749.0, delta=-542.1)
21:16:03 [INFO] [Etapa 8] PNADc no DuckDB é de 2022, não 2024 — âncora t1 indisponível.
21:16:03 [INFO] [Etapa 8] PNADc t1 indisponível — deficit_calibrado_t1 = predito sem âncora macro
21:16:03 [INFO] [Etapa 8] deficit_calibrado_t1 total=3587.8
21:16:04 [INFO] [Etapa 8] deficit_calibrado_h3_t1 salvo: 431 linhas
21:16:04 [INFO] [Etapa 8] predicao_t1_metadados salvo
{'status': 'ok', 'camadas': ['deficit_calibrado_h

---
## Etapa 9 — Validação transversal

In [69]:
import modulo_estimacao.etapa9_validacao as e9
importlib.reload(e9)
from modulo_estimacao.etapa9_validacao import validar_estimativas

conn = abrir_conexao(DB_PATH)
try:
    res9 = validar_estimativas(CODIGO_IBGE, conn, periodos=["t0"])
    print(res9)
except NotImplementedError as e:
    print(f"⏳ Etapa 9 — stub:\n{e}")
finally:
    conn.close()

21:16:04 [INFO] DuckDB aberto: d:\Doutorado\UFABC\moradinha\data\processed\al_arapiraca\al_arapiraca.duckdb
21:16:04 [INFO] [Etapa 9] Validação transversal — municipio 2700300
21:16:04 [INFO] [Etapa 9] Períodos detectados: ['t0']
21:16:04 [INFO] [Etapa 9] Módulo A — CV 5-fold setor
21:16:06 [INFO] [Etapa 9 / Módulo A] CV 5-fold: R²=0.385 | RMSE=0.0850 | bias=-0.0014 | n=222
21:16:06 [INFO] [Etapa 9] Módulo B — Moran's I
21:16:06 [INFO] [Etapa 9 / Módulo B] Moran's I = 0.0345 (p=0.249) — OK
21:16:06 [INFO] [Etapa 9] Módulo C — Consistência de escala
21:16:06 [INFO] [Etapa 9 / Módulo C] t0: soma_h3=3749.0 | pnadc=3749.0 | delta=0.00% — OK
21:16:06 [INFO] [Etapa 9] Módulo D — Comparação FJP
21:16:06 [INFO] [Etapa 9 / Módulo D] data\raw\referencias\fjp_municipios.json não encontrado — comparação FJP pulada.
21:16:06 [INFO] [Etapa 9] Módulo E — Sanity checks
21:16:06 [INFO] [Etapa 9 / Módulo E] FCU: proxy_fcu=0.118 | proxy_nfcu=0.191 | fcu>nfcu=False
21:16:06 [INFO] [Etapa 9 / Módulo E] Cor

---
## Visualização — mapa H3 de déficit

Requer Etapa 5 (t0) ou Etapa 8 (t1) completa.

In [70]:
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

conn = abrir_conexao(DB_PATH)
try:
    # Tenta carregar t0; depois t1 se disponível
    from modulo_coleta.utils.db_utils import ler_tabela_espacial
    tabelas_db = set(listar_tabelas(conn))

    gdf_h3 = None
    periodo = None
    for tabela_h3, label in [
        ("deficit_calibrado_h3_t1", f"t1 ({ANO_T1})"),
        ("deficit_calibrado_h3_t0", f"t0 ({ANO_T0})"),
        ("deficit_predito_h3_t0",   f"t0 bruto ({ANO_T0})"),
    ]:
        if tabela_h3 in tabelas_db:
            gdf_h3 = ler_tabela_espacial(conn, tabela_h3)
            periodo = label
            print(f"Usando: {tabela_h3}  ({len(gdf_h3)} hexágonos)")
            break

    if gdf_h3 is None:
        print("⏳ Nenhuma tabela de predição H3 encontrada — execute as etapas primeiro.")
    else:
        gdf_setor = ler_tabela_espacial(conn, "setores_censitarios")
        gdf_lim   = ler_tabela_espacial(conn, "limite_municipal")

        col_val = "deficit_calibrado" if "deficit_calibrado" in gdf_h3.columns else "deficit_predito"

        fig, axes = plt.subplots(1, 2, figsize=(16, 8), facecolor="#f5f5f5")
        fig.suptitle(f"Déficit habitacional por hexágono H3 — {CODIGO_IBGE} | período {periodo}",
                     fontsize=13, fontweight="bold")

        for ax, cmap, title in zip(
            axes,
            ["YlOrRd", "RdYlGn_r"],
            [f"Estimativa bruta (predição)", f"Após calibração IPF"],
        ):
            ax.set_facecolor("#e8e8e8")
            gdf_lim.to_crs("EPSG:4674").boundary.plot(ax=ax, color="#333", linewidth=1.2)
            col = col_val if col_val in gdf_h3.columns else gdf_h3.select_dtypes("number").columns[0]
            gdf_h3.to_crs("EPSG:4674").plot(
                column=col, cmap=cmap, ax=ax,
                legend=True, legend_kwds={"shrink": 0.6, "label": "déficit estimado"},
                alpha=0.85,
            )
            ax.set_title(title, fontsize=10)
            ax.axis("off")

        plt.tight_layout()
        plt.show()

except Exception as e:
    print(f"Erro na visualização: {e}")
finally:
    conn.close()

21:16:07 [INFO] DuckDB aberto: d:\Doutorado\UFABC\moradinha\data\processed\al_arapiraca\al_arapiraca.duckdb
Erro na visualização: cannot import name 'ler_tabela_espacial' from 'modulo_coleta.utils.db_utils' (d:\Doutorado\UFABC\moradinha\modulo_coleta\utils\db_utils.py)


## Exportar GeoPackage

Exporta as tabelas H3 para GPKG para visualização no QGIS.

In [71]:
from modulo_coleta.utils.db_utils import ler_tabela_espacial

GPKG_OUT = BASE_DIR / "processed" / NOME_PASTA / f"deficit_h3_{CODIGO_IBGE}.gpkg"

conn = abrir_conexao(DB_PATH)
tabelas_db = set(listar_tabelas(conn))

exportadas = []
for tabela in [
    "deficit_calibrado_h3_t0",
    "deficit_calibrado_h3_t1",
    "proxy_setor",
    "covariaveis_setor_t0",
]:
    if tabela in tabelas_db:
        gdf = ler_tabela_espacial(conn, tabela)
        gdf.to_file(GPKG_OUT, layer=tabela, driver="GPKG")
        exportadas.append(tabela)
        print(f"✅ {tabela}: {len(gdf)} registros → {GPKG_OUT.name}")

if not exportadas:
    print("⏳ Nenhuma tabela de estimação encontrada para exportar.")

conn.close()

ImportError: cannot import name 'ler_tabela_espacial' from 'modulo_coleta.utils.db_utils' (d:\Doutorado\UFABC\moradinha\modulo_coleta\utils\db_utils.py)